# Notebook 01: Exploratory Data Analysis (EDA) & Risk Profiling
**Project:** ACIS Auto-Insurance Risk Analytics & Predictive Modeling  
**Author:** Soliana Hailekiros  
**Objective:** Profile historical data shapes, clean missing structures, isolate outliers, and identify low-risk customer segments using the Claim-to-Premium ratio.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure visualization aesthetics
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

print("[INFO] Analytical libraries successfully imported.")

In [ ]:
# Path to local DVC-managed raw insurance data file
DATA_PATH = "../data/raw/insurance_data.csv"

if not os.path.exists(DATA_PATH):
    print(f"[WARNING] Check data path! Creating mock historical distribution to demonstrate baseline structure pipeline...")
    # Seed reproducible distributions mapping historical auto-insurance features
    np.random.seed(42)
    n_records = 5000
    mock_provinces = ['Gauteng', 'Western Cape', 'KwaZulu-Natal', 'Eastern Cape', 'Free State']
    
    mock_data = {
        'UnderwrittenCoverID': np.arange(1001, 1001 + n_records),
        'Province': np.random.choice(mock_provinces, size=n_records, p=[0.4, 0.25, 0.15, 0.1, 0.1]),
        'PostalCode': np.random.randint(1000, 9999, size=n_records),
        'Gender': np.random.choice(['Male', 'Female', 'Corporate'], size=n_records, p=[0.55, 0.40, 0.05]),
        'TotalPremium': np.random.exponential(scale=800, size=n_records) + 150,
        'TotalClaim': np.random.choice([0, 2500, 12000, 45000], size=n_records, p=[0.92, 0.05, 0.025, 0.005]) * np.random.rand(n_records),
        'SumInsured': np.random.normal(loc=180000, scale=60000, size=n_records).clip(20000)
    }
    df = pd.DataFrame(mock_data)
    # Intentionally inject missing values to demonstrate analytical processing routines
    df.loc[df.sample(frac=0.02).index, 'Gender'] = np.nan
else:
    df = pd.read_csv(DATA_PATH)

print(f"Data State Verification: {df.shape[0]} rows, {df.shape[1]} columns.\n")
print(df.info())
df.head()

## 1. Missing Data Profiling & Structural Integrity
Before running aggregations, we must evaluate the distribution of missing or null fields across the features.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100

missing_table = pd.DataFrame({
    'Missing Records': missing_count,
    'Percentage (%)': missing_pct
}).sort_values(by='Percentage (%)', ascending=False)

print("--- Feature Completeness Audit ---")
if missing_table['Missing Records'].sum() == 0:
    print("Zero missing fields detected. Data state is clean and dense.")
else:
    print(missing_table[missing_table['Missing Records'] > 0])

## 2. Quantitative Descriptive Statistics & Feature Scale
Evaluating the central tendencies, ranges, scales, and variation fields for core continuous insurance metrics.

In [ ]:
target_cols = ['TotalPremium', 'TotalClaim', 'SumInsured']
valid_cols = [col for col in target_cols if col in df.columns]

summary_stats = df[valid_cols].describe().T
summary_stats['skewness'] = df[valid_cols].skew()
summary_stats['kurtosis'] = df[valid_cols].kurt()

print("--- Core Insurance Continuous Variables Profile ---")
summary_stats

## 3. Segment Identification & Premium-to-Claim Risk Analytics
To identify low-risk pockets where ACIS can strategically lower premium charges to attract customers safely, we group historical tracking records across categories and evaluate the **Claim-to-Premium Ratio**. Lower ratios indicate highly stable risk margins.

In [ ]:
def generate_risk_matrix(dataframe, dimension):
    if dimension not in dataframe.columns:
        print(f"[ABORT] Axis column '{dimension}' is missing from the working schema.")
        return None
        
    matrix = dataframe.groupby(dimension).agg(
        Total_Premium_Volume=('TotalPremium', 'sum'),
        Total_Claims_Paid=('TotalClaim', 'sum'),
        Average_Claim_Severity=('TotalClaim', 'mean'),
        Policy_Exposure_Count=('TotalPremium', 'count')
    ).reset_index()
    
    # Primary evaluation ratio: Total Claims divided by Total Premiums
    matrix['Claim_to_Premium_Ratio'] = matrix['Total_Claims_Paid'] / matrix['Total_Premium_Volume']
    return matrix.sort_values(by='Claim_to_Premium_Ratio', ascending=True)

print("--- Geographic Segment Performance Profile (Sorted by Target Opportunity) ---")
if 'Province' in df.columns:
    geo_profile = generate_risk_matrix(df, 'Province')
    display(geo_profile)

## 4. Analytical Visualization Engineering
Visualizing target correlations and linear feature relationships to check for multicollinearity or clear demographic segmentation trends.

In [ ]:
plt.figure(figsize=(8, 5))
numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns
# Remove ID / Code variables to ensure meaningful feature matrix math
clean_numeric = [col for col in numeric_fields if 'ID' not in col and 'Code' not in col]

correlation_map = df[clean_numeric].corr()
sns.heatmap(correlation_map, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Continuous Feature Correlation Heatmap - ACIS Analytics Engine', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
hue_var = 'Province' if 'Province' in df.columns else None

sns.scatterplot(
    data=df, 
    x='TotalPremium', 
    y='TotalClaim', 
    hue=hue_var, 
    alpha=0.6, 
    palette='viridis'
)

plt.title('Premium Exposure vs. Historical Claim Aggregation', fontsize=14, pad=15)
plt.xlabel('Total Premium Contributed (ZAR)')
plt.ylabel('Total Historical Claim Realized (ZAR)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title=hue_var)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, y='TotalPremium', color='skyblue')
plt.title('Premium Field Variance & Outliers')
plt.ylabel('Premium Value (ZAR)')

plt.subplot(1, 2, 2)
sns.boxplot(data=df, y='TotalClaim', color='salmon')
plt.title('Claim Distribution Skew')
plt.ylabel('Claim Value (ZAR)')

plt.tight_layout()
plt.show()